# Demo Pipeline Cham Trac Nghiem Tu Dong (OMR)

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

from src.preprocessing import doc_anh, chuyen_xam, loc_nhieu
from src.transform import tim_canh, tim_goc_giay, nan_chinh_anh
from src.reader import (
    phat_hien_anchor, phan_loai_vung_roi,
    extract_exam_code_region, read_exam_code,
    extract_student_id_region, read_student_id,
    visualize_all_regions, visualize_anchors,
)
from src.grader import segment_bubbles, calculate_score, grade_from_image

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## Buoc 1: Doc anh

In [ ]:
image_path = "../data/raw/test_sheet_01.jpg"
image = doc_anh(image_path)
print(f"Kich thuoc anh: {image.shape}")

plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Anh goc")
plt.axis('off')
plt.show()

## Buoc 2: Chuyen sang anh xam

In [ ]:
gray = chuyen_xam(image)

plt.imshow(gray, cmap='gray')
plt.title("Anh xam")
plt.axis('off')
plt.show()

## Buoc 3: Khu nhieu

In [ ]:
blurred = loc_nhieu(gray, loai_loc="gaussian", kich_thuoc=5)

plt.imshow(blurred, cmap='gray')
plt.title("Anh sau khi khu nhieu")
plt.axis('off')
plt.show()

## Buoc 4: Phat hien bien

In [ ]:
edges = tim_canh(blurred, nguong_thap=50, nguong_cao=150)

plt.imshow(edges, cmap='gray')
plt.title("Anh bien (Canny)")
plt.axis('off')
plt.show()

## Buoc 5: Tim 4 goc toa giay

In [ ]:
corners = tim_goc_giay(edges, auto_detect_cropped=True)

if corners is not None:
    print(f"Tim thay 4 goc: {corners}")
else:
    print("Anh da cat san hoac khong tim thay goc")

## Buoc 6: Nan chinh anh

In [ ]:
if corners is not None:
    warped = nan_chinh_anh(image, corners, chieu_rong=800, chieu_cao=1200)
else:
    warped = cv2.resize(image, (800, 1200))

print(f"Kich thuoc anh sau nan chinh: {warped.shape}")

plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
plt.title("Anh sau khi nan chinh")
plt.axis('off')
plt.show()

## Buoc 7: Phat hien anchor va ROI

In [ ]:
try:
    anchors = phat_hien_anchor(warped)
    print(f"Tim thay {len(anchors)} anchor markers")
    
    rois = phan_loai_vung_roi(anchors, *warped.shape[:2])
    print(f"SBD: {rois['sbd']}")
    print(f"Ma de: {rois['ma_de']}")
    print(f"Dap an: {rois['dap_an']}")
    
    anh_roi = visualize_all_regions(warped, rois=rois)
    plt.imshow(cv2.cvtColor(anh_roi, cv2.COLOR_BGR2RGB))
    plt.title("Cac vung ROI")
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Loi: {e}")

## Buoc 8: Doc ma de va SBD

In [ ]:
try:
    vung_ma_de = extract_exam_code_region(warped, *rois['ma_de'])
    vung_sbd = extract_student_id_region(warped, *rois['sbd'])
    
    ma_de = "N/A"
    for method in ["otsu", "adaptive", "binary"]:
        try:
            ma_de = read_exam_code(vung_ma_de, num_digits=3, threshold_method=method)
            break
        except:
            pass
    
    sbd = "N/A"
    for method in ["otsu", "adaptive", "binary"]:
        try:
            sbd = read_student_id(vung_sbd, num_digits=6, threshold_method=method)
            break
        except:
            pass
    
    print(f"Ma de: {ma_de}")
    print(f"So bao danh: {sbd}")
except Exception as e:
    print(f"Loi: {e}")

## Buoc 9: Phan doan va cham diem

In [ ]:
with open("../data/answer_keys/all_answer_keys.json", 'r', encoding='utf-8') as f:
    all_keys = json.load(f)

if ma_de in all_keys:
    answer_key = {int(k): v for k, v in all_keys[ma_de]["answers"].items()}
    print(f"Da tim thay dap an cho ma de {ma_de}")
else:
    print(f"Khong tim thay dap an cho ma de {ma_de}")
    answer_key = {}

In [ ]:
if answer_key:
    try:
        correct_count, score, student_answers = grade_from_image(
            warped_image=warped,
            answer_key=answer_key,
            num_questions=20,
            so_bao_danh=sbd,
            ma_de=ma_de
        )
    except Exception as e:
        print(f"Loi cham diem: {e}")
else:
    print("Khong the cham diem")